<a href="https://colab.research.google.com/github/lubabasadiyanp/sentimental-analysis-of-product-review/blob/sebin/Data%20Understanding%20%26%20preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Step1:Load Dataset & preprocessing






In [3]:
import pandas as pd
from datasets import load_dataset

# Yelp
dataset = load_dataset('yelp_review_full')
df = dataset['train'].to_pandas()
df = df[['text', 'label']].sample(n=10000, random_state=42)
df['stars'] = df['label'] + 1

# Sentiment mapping
def rating_to_sentiment(r):
    if r >= 4:   return 'Positive'
    elif r == 3: return 'Neutral'
    else:        return 'Negative'

df['sentiment'] = df['stars'].apply(rating_to_sentiment)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

yelp_review_full/train-00000-of-00001.pa(…):   0%|          | 0.00/299M [00:00<?, ?B/s]

yelp_review_full/test-00000-of-00001.par(…):   0%|          | 0.00/23.5M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/650000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [4]:
pd.set_option('display.max_columns',None)
df.head(5)

,text,label,stars,sentiment
177288,"First of all i'm not a big fan of buffet, i tr...",0,1,Negative
238756,Thanks Yelp. I was looking for the words to de...,1,2,Negative
604225,Service was so-so. They were receiving a deliv...,2,3,Neutral
2838,Stamoolis Brothers is one of the Strip Distric...,2,3,Neutral
586957,I want to give a 2 stars because the service s...,0,1,Negative


In [5]:
df.info

<bound method DataFrame.info of                                                      text  label  stars  \
177288  First of all i'm not a big fan of buffet, i tr...      0      1   
238756  Thanks Yelp. I was looking for the words to de...      1      2   
604225  Service was so-so. They were receiving a deliv...      2      3   
2838    Stamoolis Brothers is one of the Strip Distric...      2      3   
586957  I want to give a 2 stars because the service s...      0      1   
...                                                   ...    ...    ...   
456268  Four Hours of my life I'd like to have back!\n...      0      1   
259135  Small stadium and would have great views all a...      1      2   
496284  I think Porktropolis has the BBQ thing nailed....      2      3   
512684  Best CU ever!!! Great customer service!!! Know...      4      5   
597788  Came in for late lunch on a Tuesday. I sat at ...      2      3   

       sentiment  
177288  Negative  
238756  Negative  
604225   Neutral  
2838     Neutral  
586957  Negative  
...          ...  
456268  Negative  
259135  Negative  
496284   Neutral  
512684  Positive  
597788   Neutral  

[10000 rows x 4 columns]>

In [6]:
df.describe()

,label,stars
count,10000.000000,10000.000000
mean,1.998100,2.998100
std,1.410849,1.410849
min,0.000000,1.000000
25%,1.000000,2.000000
50%,2.000000,3.000000
75%,3.000000,4.000000
max,4.000000,5.000000


In [7]:
df.columns

Index(['text', 'label', 'stars', 'sentiment'], dtype='object')

In [8]:

df.isnull().sum()

,0
text,0
label,0
stars,0
sentiment,0


In [9]:
df.duplicated().sum()

np.int64(0)

# Step 2:Text Cleaning

In [10]:
import re

def clean_text(text):
    text = str(text).lower().strip()
    text = re.sub(r'http\S+|www\.\S+', '', text)
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'[^\w\s!?.,\'"-]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['clean_text'] = df['text'].apply(clean_text)
df.to_csv('preprocessed_data.csv', index=False)

In [13]:
import re
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    get_linear_schedule_with_warmup
)

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, f1_score
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [15]:
df = pd.read_csv('preprocessed_data.csv')

# ── Validate expected columns ─────────────────────────────────────────────────
assert 'clean_text' in df.columns, "Missing column: 'clean_text'"
assert 'sentiment'  in df.columns, "Missing column: 'sentiment'"

df = df.dropna(subset=['clean_text', 'sentiment']).reset_index(drop=True)
df['clean_text'] = df['clean_text'].astype(str)

# Define the mapping from sentiment strings to integers
sentiment_to_int_map = {'Negative': 0, 'Neutral': 1, 'Positive': 2}
# Apply the mapping to the 'sentiment' column
df['sentiment']  = df['sentiment'].map(sentiment_to_int_map).astype(int)

label_map   = {0: 'Negative', 1: 'Neutral', 2: 'Positive'}
label_names = ['Negative', 'Neutral', 'Positive']

print(f"Loaded {len(df)} rows")
print(df['sentiment'].map(label_map).value_counts())
df.head()

Loaded 10000 rows
sentiment
Negative    4025
Positive    3995
Neutral     1980
Name: count, dtype: int64


,text,label,stars,sentiment,clean_text
0,"First of all i'm not a big fan of buffet, i tr...",0,1,0,"first of all i'm not a big fan of buffet, i tr..."
1,Thanks Yelp. I was looking for the words to de...,1,2,0,thanks yelp. i was looking for the words to de...
2,Service was so-so. They were receiving a deliv...,2,3,1,service was so-so. they were receiving a deliv...
3,Stamoolis Brothers is one of the Strip Distric...,2,3,1,stamoolis brothers is one of the strip distric...
4,I want to give a 2 stars because the service s...,0,1,0,i want to give a 2 stars because the service s...


In [16]:
X = df['clean_text'].tolist()
y = df['sentiment'].tolist()

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print(f"Train : {len(X_train):,} samples")
print(f"Val   : {len(X_val):,} samples")
print(f"Test  : {len(X_test):,} samples")

Train : 7,000 samples
Val   : 1,500 samples
Test  : 1,500 samples


In [17]:
tfidf_vectorizer = TfidfVectorizer(
    max_features=50000,
    ngram_range=(1, 2),      # unigrams + bigrams
    sublinear_tf=True,       # log normalization
    min_df=2,
    strip_accents='unicode',
    analyzer='word'
)

# Fit only on train — transform all splits
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_val_tfidf   = tfidf_vectorizer.transform(X_val)
X_test_tfidf  = tfidf_vectorizer.transform(X_test)

print(f"TF-IDF feature matrix: {X_train_tfidf.shape}")

with open('tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(tfidf_vectorizer, f)
print("Saved: tfidf_vectorizer.pkl")

TF-IDF feature matrix: (7000, 50000)
Saved: tfidf_vectorizer.pkl
